# SE-ResNet ECG — Comparative Study

**CS-598 DLH Project · April 2026**

This notebook implements and evaluates **SE-ResNet-50** (Squeeze-Excitation ResNet) as proposed in:

> Nonaka, N. & Seita, J. (2021). *In-depth Benchmarking of Deep Neural Network Architectures for ECG Diagnosis.* MLHC 2021.

**Goal:** Reproduce the paper's SE-ResNet-50 architecture within the PyHealth framework and compare it against the two baselines from `ptbxl_pyhealth_pipeline.ipynb` using the same dataset, splits, and evaluation protocol.

| Model | Architecture | Source |
|---|---|---|
| **SE-ResNet-50** | ResNet-50 1D + Squeeze-Excitation blocks | `se_resnet_ecg.py` (this study) |
| **SparcNet** | DenseNet-inspired 1D TCN | PyHealth built-in |
| **BiLSTMECG** | Bidirectional LSTM, 1 layer, hidden=64 | `PyHealth/pyhealth/models/bilstm_ecg.py` |

---

### SE-ResNet architecture (paper §4.2 + Appendix A.3)

```
Input (B, 12, T)  — 12 ECG leads × T samples
  → Conv1d(12, 64, k=7, s=2) → BN → ReLU → MaxPool1d(k=3, s=2)
  → Stage 1: 3 × Bottleneck1D(256)   + SE
  → Stage 2: 4 × Bottleneck1D(512)   + SE   [stride=2]
  → Stage 3: 6 × Bottleneck1D(1024)  + SE   [stride=2]
  → Stage 4: 3 × Bottleneck1D(2048)  + SE   [stride=2]
  → AdaptiveAvgPool1d(1) → Flatten  (B, 2048)
  → Neck: Linear(2048 → 256)          (paper: all backbones output 256-dim)
  → Head: FC(256) → ReLU → BN → Dropout(0.5) → FC(K)
  → BCEWithLogitsLoss  (multi-label)
```

**SE Block** (Hu et al., 2018 — Squeeze-and-Excitation Networks):
- *Squeeze*: ``AdaptiveAvgPool1d(1)`` → ``(B, C)``
- *Excitation*: ``Linear(C, C//16) → ReLU → Linear(C//16, C) → Sigmoid``
- *Scale*: element-wise multiply channel weights onto the feature map

Best hyperparams from paper grid search:
- SE-ResNet-50, **lr = 0.01**, **batch = 64** → ROC-AUC = 0.9082 on PTB-XL `all`

---
### ⚡ Execution guide
Run all cells top-to-bottom. Cells 3–6 reuse the pickle cache created by
`ptbxl_pyhealth_pipeline.ipynb` if it was run first; otherwise they rebuild
the cache (~2 min). The training cells use `N_SUBSAMPLE=2000` / 3 epochs for
speed. Change `N_SUBSAMPLE` and `TRAIN_EPOCHS` at the top of their cells for
publication-quality results.

---
## Cell 1 — Imports and device

In [ ]:
# ── Standard library & scientific stack ──────────────────────────────────────
import ast
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score, roc_curve

# ── PyHealth ──────────────────────────────────────────────────────────────────
from pyhealth.datasets import (
    BaseSignalDataset,
    SampleSignalDataset,
    get_dataloader,
    split_by_patient,
)
import pyhealth.datasets as _ph_ds
from pyhealth.models import BaseModel, SparcNet
from pyhealth.trainer import Trainer
from pyhealth.metrics import multilabel_metrics_fn

# ── SE-ResNet-50 model (defined in se_resnet_ecg.py, same folder) ─────────────
_CS598_DIR = Path(".").resolve()
if str(_CS598_DIR) not in sys.path:
    sys.path.insert(0, str(_CS598_DIR))
from se_resnet_ecg import SEResNetECG

# ── BiLSTMECG (from PyHealth models folder) ───────────────────────────────────
_PYHEALTH_MODELS = _CS598_DIR.parent / "pyhealth" / "models"
if str(_PYHEALTH_MODELS) not in sys.path:
    sys.path.insert(0, str(_PYHEALTH_MODELS))
from bilstm_ecg import BiLSTMECG

from scipy.io import loadmat as scipy_loadmat
warnings.filterwarnings("ignore", category=FutureWarning, module="dask")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = (
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)

print("✅  All imports successful")
print(f"   PyTorch  : {torch.__version__}")
print(f"   Device   : {DEVICE}")
try:
    import pyhealth; print(f"   PyHealth : {pyhealth.__version__}")
except Exception:
    print("   PyHealth : (version not exposed)")
print(f"   SEResNetECG from : {Path(SEResNetECG.__module__.replace('.', '/') + '.py').name}")

---
## Cell 2 — Dataset setup

Identical to `ptbxl_pyhealth_pipeline.ipynb` Cells 3–5. Reuses the
`ptbxl-pyhealth.csv` metadata file if already present.

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
_PROJECT_ROOT  = Path("/Users/anuragd/CS-598_HealthCareAssignment/DLH_Project/PyHealth/cs598_project")
CINC_PTBXL_ROOT = (
    _PROJECT_ROOT.parent.parent /
    "classification-of-12-lead-ecgs-the-physionetcomputing-in-cardiology-challenge-2020-1.0.2"
    / "training" / "ptb-xl"
)
print(f"CinC PTB-XL root : {CINC_PTBXL_ROOT}")
assert CINC_PTBXL_ROOT.exists(), f"Data not found at {CINC_PTBXL_ROOT}"

# ── prepare_metadata() — same as main pipeline ────────────────────────────────
def prepare_metadata(root: Path) -> Path:
    """Scan g*/  .hea files → write ptbxl-pyhealth.csv (jtwells2 logic)."""
    cache_dir  = Path.home() / ".cache" / "pyhealth" / "ptbxl"
    shared_csv = root / "ptbxl-pyhealth.csv"
    cache_csv  = cache_dir / "ptbxl-pyhealth.csv"
    if shared_csv.exists():
        print(f"  CSV found: {shared_csv}"); return shared_csv
    if cache_csv.exists():
        print(f"  CSV cache: {cache_csv}"); return cache_csv
    rows = []
    for gdir in sorted(root.glob("g*/")):
        if not gdir.is_dir(): continue
        for hea in sorted(gdir.glob("*.hea")):
            mat = hea.with_suffix(".mat")
            if not mat.exists(): continue
            rid = hea.stem
            age, sex, scp_codes = None, None, ""
            for raw in hea.read_text(encoding="utf-8", errors="replace").splitlines():
                line = raw.strip()
                if line.startswith("# Age:"):
                    try: age = int(float(line.split(":", 1)[-1].strip()))
                    except: pass
                elif line.startswith("# Sex:"):  sex = line.split(":", 1)[-1].strip()
                elif line.startswith("# Dx:"): scp_codes = line.split(":", 1)[-1].strip()
            rows.append({"patient_id": rid, "record_id": rid,
                         "signal_file": str(mat.absolute()),
                         "age": age, "sex": sex, "scp_codes": scp_codes,
                         "sampling_rate": 500, "num_samples": 5000,
                         "num_leads": 12, "group": gdir.name})
    if not rows:
        raise RuntimeError(f"No .hea files found under {root}")
    df = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    try:
        df.to_csv(shared_csv, index=False); return shared_csv
    except (PermissionError, OSError):
        cache_dir.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache_csv, index=False); return cache_csv

csv_path = prepare_metadata(CINC_PTBXL_ROOT)
df_meta  = pd.read_csv(csv_path)
print(f"  Metadata: {df_meta.shape[0]:,} records")

# ── PTBXLDataset (BaseSignalDataset adapter — jtwells2 schema) ────────────────
class PTBXLDataset(BaseSignalDataset):
    def __init__(self, csv_path, **kwargs):
        self._csv_path = Path(csv_path)
        super().__init__(root=str(self._csv_path.parent),
                         dataset_name="ptbxl", **kwargs)
    def process_EEG_data(self):
        df = pd.read_csv(self._csv_path)
        patients = defaultdict(list)
        for _, row in df.iterrows():
            pid = str(row["patient_id"])
            patients[pid].append({
                "patient_id": pid, "ecg_id": str(row["record_id"]),
                "mat_path":   str(row["signal_file"]),
                "scp_codes":  str(row["scp_codes"]) if pd.notna(row["scp_codes"]) else "",
                "age": row.get("age"), "sex": row.get("sex"),
            })
        return dict(patients)

print("\nInitialising PTBXLDataset adapter …")
dataset = PTBXLDataset(csv_path=str(csv_path))
print(f"  Total records in dataset : {sum(len(v) for v in dataset.patients.values()):,}")

---
## Cell 3 — Task function: PTB-XL superdiagnostic (5 classes)

Same SNOMED_TO_SUPERDIAG mapping and task logic as the main pipeline.
Pickle cache is reused if present.

In [ ]:
# ── SNOMED → superdiagnostic class mapping (jtwells2 :: ptbxl_multilabel_classification.py) ──
SNOMED_TO_SUPERDIAG = {
    "426783006": "NORM",
    "57054005":  "MI",   "164865005": "MI",   "413444003": "MI",  "413867000": "MI",
    "164861001": "MI",   "164857002": "MI",   "164860000": "MI",  "164864009": "MI",
    "164867002": "MI",
    "164931005": "STTC", "164934002": "STTC", "59931005":  "STTC", "164947007": "STTC",
    "164917005": "STTC", "251268003": "STTC", "428750005": "STTC",
    "270492004": "CD",   "195042002": "CD",   "27885002":  "CD",  "6374002":   "CD",
    "713427006": "CD",   "713426002": "CD",   "164909002": "CD",  "59118001":  "CD",
    "698252002": "CD",   "445118002": "CD",   "10370003":  "CD",  "164889003": "CD",
    "164890007": "CD",   "426627000": "CD",   "427393009": "CD",  "426177001": "CD",
    "427084000": "CD",   "63593006":  "CD",   "17338001":  "CD",  "284470004": "CD",
    "427172004": "CD",
    "55827005":         "HYP", "446358003":       "HYP", "73282002":  "HYP",
    "67751000119106":   "HYP", "446813000":       "HYP", "39732003":  "HYP",
    "47665007":         "HYP", "251146004":       "HYP",
}
SUPERDIAG_CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

# ── Task function — mirrors jtwells2 PTBXLMultilabelClassification.__call__() ─
def ptbxl_superdiagnostic_fn(record_list, sampling_rate=100, native_freq=500):
    samples     = []
    decimate    = native_freq // sampling_rate
    expected    = 5000 // decimate
    cache_dir   = Path.home() / ".cache" / "pyhealth_ptbxl" / f"cinc_{sampling_rate}hz"
    cache_dir.mkdir(parents=True, exist_ok=True)
    for visit in record_list:
        pid     = visit["patient_id"]
        ecg_id  = visit["ecg_id"]
        raw     = str(visit["scp_codes"])
        codes   = [c.strip() for c in raw.split(",") if c.strip()]
        labels  = list({SNOMED_TO_SUPERDIAG[c] for c in codes if c in SNOMED_TO_SUPERDIAG})
        if not labels: continue
        mat_path = visit["mat_path"]
        if mat_path and Path(mat_path).exists():
            mat    = scipy_loadmat(mat_path)
            signal = mat["val"].astype(np.float32) / 200.0
        else:
            t      = np.linspace(0, 10, native_freq * 10)
            lead   = 0.5*np.sin(2*np.pi*1.2*t) + np.random.randn(native_freq*10)*0.04
            signal = np.tile(lead, (12, 1)).astype(np.float32)
        signal = signal[:, ::decimate]
        if signal.shape[1] < expected: continue
        pkl_path = str(cache_dir / f"{ecg_id}.pkl")
        if not Path(pkl_path).exists():
            with open(pkl_path, "wb") as f:
                pickle.dump({"signal": signal}, f)
        samples.append({"patient_id": pid, "record_id": ecg_id,
                         "ecg_id": ecg_id, "epoch_path": pkl_path, "labels": labels})
    return samples

print("Building SampleSignalDataset (uses cached pickles if available) …")
sample_dataset = dataset.set_task(ptbxl_superdiagnostic_fn)
sample_dataset.input_info["labels"] = {"type": str, "dim": 2}

n_total = len(sample_dataset)
print(f"  Total samples : {n_total:,}  (one per ECG recording)")
print(f"  input_info    : {sample_dataset.input_info}")

# Label prevalence
pos_rates = np.zeros(len(SUPERDIAG_CLASSES))
for s in sample_dataset.samples:
    for cls in s["labels"]:
        if cls in SUPERDIAG_CLASSES:
            pos_rates[SUPERDIAG_CLASSES.index(cls)] += 1
pos_rates /= max(n_total, 1)
print("\nLabel prevalence:")
for cls, rate in zip(SUPERDIAG_CLASSES, pos_rates):
    print(f"  {cls:<6} {rate*100:5.1f}%  {'█'*int(rate*30)}")

---
## Cell 4 — Patient-level split + DataLoaders

Same 80/10/10 split with seed=42 as the main pipeline. `N_SUBSAMPLE=2000`
matches the ultra-fast test in `ptbxl_pyhealth_pipeline.ipynb`.

In [ ]:
from torch.utils.data import DataLoader as _DL, Subset as _Subset

# ⚡ Change to 10_000 or None for publication-quality results
N_SUBSAMPLE  = 2_000
BATCH_SIZE   = 32

n_full  = len(sample_dataset.samples)
rng     = np.random.default_rng(seed=42)
all_idx = sorted(
    rng.choice(n_full, size=min(N_SUBSAMPLE, n_full), replace=False).tolist()
) if N_SUBSAMPLE and N_SUBSAMPLE < n_full else list(range(n_full))
print(f"⚡ Working with {len(all_idx):,} records "
      f"({'subsampled' if len(all_idx) < n_full else 'full dataset'}, seed=42).")

# Patient-level split (80/10/10)
pid_to_pos = {}
for pos, idx in enumerate(all_idx):
    pid = sample_dataset.samples[idx]["patient_id"]
    pid_to_pos.setdefault(pid, []).append(pos)
pids = list(pid_to_pos.keys())
np.random.default_rng(42).shuffle(pids)
n_p  = len(pids)
n_tr = int(0.80 * n_p)
n_vl = int(0.10 * n_p)
tr_pos = [p for pid in pids[:n_tr]           for p in pid_to_pos[pid]]
vl_pos = [p for pid in pids[n_tr:n_tr+n_vl]  for p in pid_to_pos[pid]]
te_pos = [p for pid in pids[n_tr+n_vl:]       for p in pid_to_pos[pid]]
tr_idx = [all_idx[p] for p in tr_pos]
vl_idx = [all_idx[p] for p in vl_pos]
te_idx = [all_idx[p] for p in te_pos]

train_ds = _Subset(sample_dataset, tr_idx)
val_ds   = _Subset(sample_dataset, vl_idx)
test_ds  = _Subset(sample_dataset, te_idx)

train_loader = _DL(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                   collate_fn=_ph_ds.collate_fn_dict)
val_loader   = _DL(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                   collate_fn=_ph_ds.collate_fn_dict)
test_loader  = _DL(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                   collate_fn=_ph_ds.collate_fn_dict)

print(f"\nPatient-level split (80/10/10, seed=42):")
print(f"  Train : {len(tr_idx):>5,} samples  | {len(train_loader)} batches")
print(f"  Val   : {len(vl_idx):>5,} samples")
print(f"  Test  : {len(te_idx):>5,} samples")
print(f"  Batch : {BATCH_SIZE}")

# Leak check
tr_pids = {sample_dataset.samples[i]["patient_id"] for i in tr_idx}
vl_pids = {sample_dataset.samples[i]["patient_id"] for i in vl_idx}
te_pids = {sample_dataset.samples[i]["patient_id"] for i in te_idx}
assert not (tr_pids & vl_pids), "LEAKAGE: train ∩ val!"
assert not (tr_pids & te_pids), "LEAKAGE: train ∩ test!"
assert not (vl_pids & te_pids), "LEAKAGE: val  ∩ test!"
print("\n✅  Zero patient overlap across splits.")

---
## Cell 5 — Instantiate all three models

All models share the same `dataset`, `feature_keys`, `label_key`, and `mode`
so PyHealth's `BaseModel` wires loss, label tokenizer, and metrics identically.

In [ ]:
# ── Shared kwargs for every model ────────────────────────────────────────────
_common = dict(
    dataset      = sample_dataset,
    feature_keys = ["signal"],
    label_key    = "labels",
    mode         = "multilabel",
)

# ── Model A: SparcNet (PyHealth built-in) ─────────────────────────────────────
sparcnet = SparcNet(**_common)
n_sparcnet = sum(p.numel() for p in sparcnet.parameters() if p.requires_grad)

# ── Model B: BiLSTMECG (paper best: d1_h64) ───────────────────────────────────
bilstm = BiLSTMECG(**_common, hidden_size=64, n_layers=1)
n_bilstm = sum(p.numel() for p in bilstm.parameters() if p.requires_grad)

# ── Model C: SE-ResNet-50 (proposed, paper-aligned) ───────────────────────────
# Best paper config: lr=0.01, batch=64  (grid search Table 5a, Appendix A.3)
# Architecture: ResNet-50 1D + SE blocks + backbone_out=256 + dropout=0.5
se_resnet = SEResNetECG(
    **_common,
    layers       = [3, 4, 6, 3],   # SE-ResNet-50
    se_reduction = 16,
    backbone_out = 256,
    dropout      = 0.5,
)
n_se_resnet = sum(p.numel() for p in se_resnet.parameters() if p.requires_grad)

# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 68)
print("  MODEL SUMMARY")
print("=" * 68)
print(f"  {'Model':<20}  {'Params':>10}  {'Architecture'}")
print("  " + "─" * 62)
print(f"  {'SparcNet':<20}  {n_sparcnet:>10,}  DenseNet-inspired 1D TCN (PyHealth)")
print(f"  {'BiLSTMECG (d1_h64)':<20}  {n_bilstm:>10,}  Bidirectional LSTM, 1 layer, h=64")
print(f"  {'SE-ResNet-50':<20}  {n_se_resnet:>10,}  ResNet-50 1D + SE blocks (proposed)")
print("=" * 68)
print(f"\n  Input  : (B, 12, 1000)  — 12 leads × 1000 samples @ 100 Hz")
print(f"  Output : (B, {len(SUPERDIAG_CLASSES)})  — {SUPERDIAG_CLASSES}")
print(f"  Loss   : BCEWithLogitsLoss (multi-label)")

# Smoke test — verify all three models run a forward pass
test_batch = next(iter(train_loader))
with torch.no_grad():
    out_s  = sparcnet(**test_batch)
    out_b  = bilstm(**test_batch)
    out_se = se_resnet(**test_batch)
print(f"\nSmoke test (batch={BATCH_SIZE}):")
print(f"  SparcNet   logit={tuple(out_s['logit'].shape)}   loss={out_s['loss'].item():.4f}")
print(f"  BiLSTMECG  logit={tuple(out_b['logit'].shape)}   loss={out_b['loss'].item():.4f}")
print(f"  SE-ResNet  logit={tuple(out_se['logit'].shape)}  loss={out_se['loss'].item():.4f}")
print("\n✅  All three models forward pass successful.")

---
## Cell 6 — Train SE-ResNet-50

Paper best config: **lr = 0.01**, batch = 64 (grid search Table 5a).  
We use batch=32 here to match the baselines. Change `TRAIN_EPOCHS` for
publication-quality convergence (paper uses 250 with early stopping).

In [ ]:
TRAIN_EPOCHS = 3   # ⚡ ultra-fast test; change to 25+ for quality results
SE_RESNET_LR = 0.01  # paper best hyperparameter (grid search Table 5a)

print("=" * 60)
print(f"  Training SE-ResNet-50  [{TRAIN_EPOCHS} epochs, N={len(tr_idx)} samples]")
print(f"  lr = {SE_RESNET_LR}  (paper best from grid search)")
print("=" * 60)

trainer_se = Trainer(
    model          = se_resnet,
    metrics        = ["roc_auc_macro", "f1_macro"],
    enable_logging = True,
    output_path    = "./output/se_resnet",
    exp_name       = "seresnet50_superdiag_5class",
)

trainer_se.train(
    train_dataloader        = train_loader,
    val_dataloader          = val_loader,
    epochs                  = TRAIN_EPOCHS,
    optimizer_params        = {"lr": SE_RESNET_LR},
    weight_decay            = 1e-4,
    monitor                 = "roc_auc_macro",
    monitor_criterion       = "max",
    load_best_model_at_last = True,
)

print("\nSE-ResNet-50 training complete.")

---
## Cell 7 — Evaluate SE-ResNet-50

In [ ]:
print("=" * 60)
print("  SE-ResNet-50 — Test Set Evaluation")
print("=" * 60)

se_scores = trainer_se.evaluate(test_loader)
print(f"\n  Test ROC-AUC (macro) : {se_scores['roc_auc_macro']:.4f}")
print(f"  Test F1   (macro)    : {se_scores['f1_macro']:.4f}")
print(f"  Test Loss            : {se_scores['loss']:.4f}")

# Per-class breakdown
y_true_se, y_prob_se, _ = trainer_se.inference(test_loader)

per_class_auc_se = {}
for k, cls in enumerate(SUPERDIAG_CLASSES):
    col = y_true_se[:, k]
    if col.sum() > 0 and (1 - col).sum() > 0:
        per_class_auc_se[cls] = roc_auc_score(col, y_prob_se[:, k])
    else:
        per_class_auc_se[cls] = float("nan")

print(f"\n  Per-class ROC-AUC:")
for cls, auc in per_class_auc_se.items():
    bar = "█" * int((auc if not np.isnan(auc) else 0) * 20)
    tag = f"{auc:.4f}" if not np.isnan(auc) else "  N/A"
    print(f"    {cls:<6}  {tag}  {bar}")

# ROC curves
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for k, (cls, color) in enumerate(zip(SUPERDIAG_CLASSES, colors)):
    col = y_true_se[:, k]
    if col.sum() > 0 and (1-col).sum() > 0:
        fpr, tpr, _ = roc_curve(col, y_prob_se[:, k])
        axes[0].plot(fpr, tpr, color=color, lw=1.8,
                     label=f"{cls} (AUC={per_class_auc_se[cls]:.3f})")
axes[0].plot([0,1],[0,1],"k--",lw=1,alpha=0.5,label="Random")
axes[0].set_title("SE-ResNet-50 — Per-class ROC Curves", fontweight="bold")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].legend(fontsize=9)

per_class_f1_se = {}
for k, cls in enumerate(SUPERDIAG_CLASSES):
    col  = y_true_se[:, k]
    pred = (y_prob_se[:, k] >= 0.5).astype(int)
    per_class_f1_se[cls] = f1_score(col, pred, zero_division=0)
axes[1].bar(SUPERDIAG_CLASSES, [per_class_f1_se[c] for c in SUPERDIAG_CLASSES],
            color=colors, edgecolor="white")
for i, (cls, sc) in enumerate(per_class_f1_se.items()):
    axes[1].text(i, sc+0.01, f"{sc:.3f}", ha="center", fontsize=9)
axes[1].set_ylim(0, 1.15)
axes[1].set_title("SE-ResNet-50 — Per-class F1 Score", fontweight="bold")
axes[1].set_ylabel("F1 Score")
plt.tight_layout(); plt.show()

---
## Cell 8 — Train baseline models (SparcNet + BiLSTMECG)

Identical training setup as `ptbxl_pyhealth_pipeline.ipynb` for fair comparison.

In [ ]:
BASELINE_LR = 1e-3   # matches main pipeline

# ── SparcNet ──────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"  Training SparcNet  [{TRAIN_EPOCHS} epochs, N={len(tr_idx)}]")
print("=" * 60)

trainer_sparcnet = Trainer(
    model=sparcnet, metrics=["roc_auc_macro", "f1_macro"],
    enable_logging=True, output_path="./output/se_resnet",
    exp_name="sparcnet_cmp_superdiag_5class",
)
trainer_sparcnet.train(
    train_dataloader=train_loader, val_dataloader=val_loader,
    epochs=TRAIN_EPOCHS, optimizer_params={"lr": BASELINE_LR},
    weight_decay=1e-4, monitor="roc_auc_macro", monitor_criterion="max",
    load_best_model_at_last=True,
)
sparcnet_scores = trainer_sparcnet.evaluate(test_loader)
print(f"\n  SparcNet  AUC={sparcnet_scores['roc_auc_macro']:.4f}  "
      f"F1={sparcnet_scores['f1_macro']:.4f}")

# ── BiLSTMECG ─────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print(f"  Training BiLSTMECG  [{TRAIN_EPOCHS} epochs, N={len(tr_idx)}]")
print("=" * 60)

trainer_bilstm = Trainer(
    model=bilstm, metrics=["roc_auc_macro", "f1_macro"],
    enable_logging=True, output_path="./output/se_resnet",
    exp_name="bilstm_cmp_superdiag_5class",
)
trainer_bilstm.train(
    train_dataloader=train_loader, val_dataloader=val_loader,
    epochs=TRAIN_EPOCHS, optimizer_params={"lr": BASELINE_LR},
    weight_decay=1e-4, monitor="roc_auc_macro", monitor_criterion="max",
    load_best_model_at_last=True,
)
bilstm_scores = trainer_bilstm.evaluate(test_loader)
print(f"\n  BiLSTMECG AUC={bilstm_scores['roc_auc_macro']:.4f}  "
      f"F1={bilstm_scores['f1_macro']:.4f}")

---
## Cell 9 — Three-model comparative results

Side-by-side ROC-AUC and F1 comparison of all three models on the same
test set, with per-class bar charts for SE-ResNet.

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
all_scores = {
    "SparcNet":      sparcnet_scores,
    "BiLSTMECG":     bilstm_scores,
    "SE-ResNet-50":  se_scores,
}
model_colors = {"SparcNet": "#2196F3", "BiLSTMECG": "#FF5722", "SE-ResNet-50": "#4CAF50"}

print("\n" + "=" * 68)
print("  COMPARATIVE RESULTS — 3-Model Ablation")
print(f"  Dataset   : PTB-XL superdiagnostic (5 classes), N={N_SUBSAMPLE:,}")
print(f"  Protocol  : Patient-level 80/10/10 split, seed=42, {TRAIN_EPOCHS} epochs")
print("=" * 68)
print(f"  {'Model':<20}  {'ROC-AUC (macro)':>15}  {'F1 (macro)':>10}  {'Loss':>8}")
print("  " + "─" * 60)
for model_name, sc in all_scores.items():
    auc  = sc.get("roc_auc_macro", float("nan"))
    f1   = sc.get("f1_macro",      float("nan"))
    loss = sc.get("loss",          float("nan"))
    print(f"  {model_name:<20}  {auc:>15.4f}  {f1:>10.4f}  {loss:>8.4f}")
print("=" * 68)

auc_vals  = [sparcnet_scores["roc_auc_macro"], bilstm_scores["roc_auc_macro"],
             se_scores["roc_auc_macro"]]
f1_vals   = [sparcnet_scores["f1_macro"],      bilstm_scores["f1_macro"],
             se_scores["f1_macro"]]
best = list(all_scores.keys())[int(np.argmax(auc_vals))]
print(f"\n  Best model by ROC-AUC : {best}")

# ── Comparative bar charts ────────────────────────────────────────────────────
model_names = ["SparcNet", "BiLSTMECG", "SE-ResNet-50"]
colors_list = [model_colors[m] for m in model_names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"3-Model Comparison: SparcNet vs BiLSTMECG vs SE-ResNet-50\n"
    f"PTB-XL superdiagnostic (5 classes) | N={N_SUBSAMPLE:,} samples | "
    f"{TRAIN_EPOCHS} epochs",
    fontsize=11, fontweight="bold"
)

x = np.arange(len(model_names))
axes[0].bar(x, auc_vals, color=colors_list, edgecolor="white", width=0.55)
axes[0].axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Random")
for i, (m, v) in enumerate(zip(model_names, auc_vals)):
    axes[0].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=10)
axes[0].set_ylim(0, 1.0); axes[0].set_ylabel("ROC-AUC (macro)", fontsize=11)
axes[0].set_title("Test ROC-AUC", fontweight="bold")
axes[0].legend(fontsize=9)

axes[1].bar(x, f1_vals, color=colors_list, edgecolor="white", width=0.55)
for i, (m, v) in enumerate(zip(model_names, f1_vals)):
    axes[1].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")
axes[1].set_xticks(x); axes[1].set_xticklabels(model_names, rotation=10)
axes[1].set_ylim(0, 1.0); axes[1].set_ylabel("F1 Score (macro)", fontsize=11)
axes[1].set_title("Test F1 Score", fontweight="bold")

plt.tight_layout()
plt.savefig("ptbxl_3model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nFigure saved → ptbxl_3model_comparison.png")

---
## Cell 10 — Per-class comparison across all three models

In [ ]:
# Collect per-class AUC for all models
y_true_sn, y_prob_sn, _ = trainer_sparcnet.inference(test_loader)
y_true_bl, y_prob_bl, _ = trainer_bilstm.inference(test_loader)
# SE-ResNet inference already done (y_true_se, y_prob_se)

all_inferences = {
    "SparcNet":     (y_true_sn, y_prob_sn),
    "BiLSTMECG":    (y_true_bl, y_prob_bl),
    "SE-ResNet-50": (y_true_se, y_prob_se),
}

# Build per-class AUC table
per_class_auc_all = {}
for model_name, (y_true, y_prob) in all_inferences.items():
    per_class_auc_all[model_name] = {}
    for k, cls in enumerate(SUPERDIAG_CLASSES):
        col = y_true[:, k]
        if col.sum() > 0 and (1 - col).sum() > 0:
            per_class_auc_all[model_name][cls] = roc_auc_score(col, y_prob[:, k])
        else:
            per_class_auc_all[model_name][cls] = float("nan")

# Print table
print("Per-class ROC-AUC:")
print(f"  {'Class':<8}", end="")
for m in all_inferences.keys():
    print(f"  {m:>14}", end="")
print()
print("  " + "─" * 54)
for cls in SUPERDIAG_CLASSES:
    print(f"  {cls:<8}", end="")
    for m in all_inferences.keys():
        v = per_class_auc_all[m][cls]
        print(f"  {v:>14.4f}" if not np.isnan(v) else f"  {'  N/A':>14}", end="")
    print()

# Grouped bar chart per class
x   = np.arange(len(SUPERDIAG_CLASSES))
w   = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
for i, (model_name, color) in enumerate(model_colors.items()):
    vals = [per_class_auc_all[model_name][cls] for cls in SUPERDIAG_CLASSES]
    vals = [0 if np.isnan(v) else v for v in vals]
    ax.bar(x + (i - 1) * w, vals, width=w, label=model_name, color=color, edgecolor="white")

ax.set_xticks(x); ax.set_xticklabels(SUPERDIAG_CLASSES)
ax.set_ylim(0, 1.1); ax.set_ylabel("ROC-AUC", fontsize=11)
ax.set_title("Per-class ROC-AUC: SparcNet vs BiLSTMECG vs SE-ResNet-50",
             fontweight="bold")
ax.axhline(0.5, color="grey", linestyle="--", linewidth=0.8, label="Random")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("ptbxl_per_class_auc_3model.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → ptbxl_per_class_auc_3model.png")

---
## Cell 11 — Reproducibility checklist

In [ ]:
import sys, torch, numpy, sklearn, pandas

print("=" * 60)
print("  REPRODUCIBILITY CHECKLIST")
print("=" * 60)
print(f"  Python       : {sys.version.split()[0]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  NumPy        : {numpy.__version__}")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  pandas       : {pandas.__version__}")
print("-" * 60)
print(f"  Random seed  : 42")
print(f"  N_SUBSAMPLE  : {N_SUBSAMPLE:,}")
print(f"  TRAIN_EPOCHS : {TRAIN_EPOCHS}")
print(f"  Batch size   : {BATCH_SIZE}  (SE-ResNet paper best: 64)")
print(f"  SE-ResNet LR : {SE_RESNET_LR}  (paper best from grid search Table 5a)")
print(f"  Baseline LR  : {BASELINE_LR}")
print(f"  Split        : 80/10/10 patient-level")
print(f"  Loss         : BCEWithLogitsLoss (multi-label)")
print(f"  Metrics      : ROC-AUC macro, F1 macro (thr=0.5)")
print(f"  Device       : {DEVICE}")
print("=" * 60)
print()
print("  SE-ResNet-50 architecture (se_resnet_ecg.py):")
print(f"    layers       = [3, 4, 6, 3]  (ResNet-50)")
print(f"    se_reduction = 16  (SE-Net paper default)")
print(f"    backbone_out = 256  (per paper: all backbones output 256)")
print(f"    dropout      = 0.5")
print()
print("  Reference:")
print("    Nonaka & Seita (2021). In-depth Benchmarking of Deep Neural")
print("    Network Architectures for ECG Diagnosis. MLHC 2021.")
print("=" * 60)
print("\n✅  This notebook is self-contained.")